# Stage 9: ROAD Adversarial Defence

Trains two adversarially-trained (defended) CNNs and compares them against the
undefended baseline under the PGD attack sweep, per class.

Two defence configurations:
- MEANINGFUL: adversarial training at eps [0.01, 0.05, 0.10], the band where
  classes transition from robust to collapsed.
- FULL: adversarial training at eps [0.01, 0.05, 0.10, 0.20, 0.30], matching the
  CICIoV2024 methodology (includes high-eps examples).

Comparing the two tests whether high-epsilon training examples help or hurt.
Static adversarial training (adversarial set crafted once from the base model),
augmenting the clean set rather than replacing it.

Prediction from the distance mechanism: defence should help most where
distance-to-benign is smallest (reverse-light) and little where largest
(fuzzing, speedometer).

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

import config
import models
import attacks
import defense

import numpy as np
import joblib
import torch
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Load processed ROAD arrays and encoder from notebook 06
arrays = np.load(config.PROCESSED_DIR / "road_stage2_arrays.npz")
X_train, y_train = arrays["X_train"], arrays["y_train"]
X_test,  y_test  = arrays["X_test"],  arrays["y_test"]
road_encoder = joblib.load(config.PROCESSED_DIR / "road_label_encoder.joblib")
class_names = list(road_encoder.classes_)

# Per-class balanced weights for training (same approach as the CV)
w = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(w, dtype=torch.float32, device=DEVICE)

print("X_train:", X_train.shape, " classes:", class_names)

Device: cuda
X_train: (31886, 9)  classes: ['benign', 'fuzzing', 'max-speedometer', 'reverse-light-off', 'reverse-light-on']


## Train baseline and two defended CNNs

The baseline is a plain CNN. Each defended CNN is trained via static adversarial
augmentation at its epsilon range. Adversarial training internally trains a base
model, crafts the adversarial set from it, then trains a fresh defended model on
the augmented data, so each defended model involves two training runs.

In [3]:
# --- Undefended baseline CNN (reference) ---
print(">>> Baseline CNN")
baseline = models.CNN1D(n_features=X_train.shape[1], n_classes=len(class_names))
baseline = models.train_cnn(
    baseline, X_train, y_train, n_epochs=50, device=DEVICE,
    class_weights=class_weights, random_seed=config.RANDOM_SEED,
)

# --- Defended CNN 1: meaningful range [0.01, 0.05, 0.10] ---
print("\n>>> Defended CNN (meaningful range)")
def_meaningful = defense.adversarial_train_cnn(
    X_train, y_train, strategy="pgd",
    epsilons=config.ROAD_DEFENCE_EPS_MEANINGFUL,
    n_epochs=50, device=DEVICE,
    class_weights=class_weights, random_seed=config.RANDOM_SEED,
)

# --- Defended CNN 2: full range [0.01 ... 0.30] ---
print("\n>>> Defended CNN (full range)")
def_full = defense.adversarial_train_cnn(
    X_train, y_train, strategy="pgd",
    epsilons=config.ROAD_DEFENCE_EPS_FULL,
    n_epochs=50, device=DEVICE,
    class_weights=class_weights, random_seed=config.RANDOM_SEED,
)

print("\nAll three models trained.")

>>> Baseline CNN
    epoch   1/50     loss 0.3817
    epoch   5/50     loss 0.0469
    epoch  10/50     loss 0.0202
    epoch  15/50     loss 0.0193
    epoch  20/50     loss 0.0139
    epoch  25/50     loss 0.0121
    epoch  30/50     loss 0.0165
    epoch  35/50     loss 0.0090
    epoch  40/50     loss 0.0132
    epoch  45/50     loss 0.0089
    epoch  50/50     loss 0.0069

>>> Defended CNN (meaningful range)
    epoch   1/50     loss 0.3946
    epoch   5/50     loss 0.0331
    epoch  10/50     loss 0.0181
    epoch  15/50     loss 0.0192
    epoch  20/50     loss 0.0140
    epoch  25/50     loss 0.0198
    epoch  30/50     loss 0.0104
    epoch  35/50     loss 0.0057
    epoch  40/50     loss 0.0146
    epoch  45/50     loss 0.0065
    epoch  50/50     loss 0.0043


PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

    augmented trainset: 31886 clean -> 127544 total (pgd strategy)
    epoch   1/50     loss 0.2115
    epoch   5/50     loss 0.0355
    epoch  10/50     loss 0.0139
    epoch  15/50     loss 0.0124
    epoch  20/50     loss 0.0127
    epoch  25/50     loss 0.0076
    epoch  30/50     loss 0.0080
    epoch  35/50     loss 0.0061
    epoch  40/50     loss 0.0067
    epoch  45/50     loss 0.0075
    epoch  50/50     loss 0.0058

>>> Defended CNN (full range)
    epoch   1/50     loss 0.3947
    epoch   5/50     loss 0.0354
    epoch  10/50     loss 0.0195
    epoch  15/50     loss 0.0186
    epoch  20/50     loss 0.0093
    epoch  25/50     loss 0.0098
    epoch  30/50     loss 0.0110
    epoch  35/50     loss 0.0057
    epoch  40/50     loss 0.0131
    epoch  45/50     loss 0.0060
    epoch  50/50     loss 0.0047


...

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

    augmented trainset: 31886 clean -> 191316 total (pgd strategy)
    epoch   1/50     loss 0.1737
    epoch   5/50     loss 0.0214
    epoch  10/50     loss 0.0137
    epoch  15/50     loss 0.0090
    epoch  20/50     loss 0.0069
    epoch  25/50     loss 0.0073
    epoch  30/50     loss 0.0090
    epoch  35/50     loss 0.0058
    epoch  40/50     loss 0.0062
    epoch  45/50     loss 0.0081
    epoch  50/50     loss 0.0053

All three models trained.


## Evaluate all three models under the PGD sweep, per class

Crafts PGD examples against each model's OWN predictions (white-box, the honest
test of a defence) and reports per-class F1 across the epsilon sweep. The
comparison shows whether adversarial training recovers robustness, and whether
the full range beats the meaningful range or the high-eps examples hurt.

In [4]:
from sklearn.metrics import f1_score

diversity = {
    "benign": 21188, "max-speedometer": 10559, "reverse-light-on": 5994,
    "reverse-light-off": 1525, "fuzzing": 592,
}
X_test_f = X_test.astype(np.float32)
n_classes = len(class_names)

# Wrap each model for ART so attacks are crafted against that specific model
models_to_eval = {
    "baseline":    baseline,
    "def_meaning": def_meaningful,
    "def_full":    def_full,
}

# For each model: clean per-class F1, then per-class F1 under PGD at each eps.
# White-box: each model attacked with examples crafted against itself.
def per_class_under_pgd(model):
    clf = attacks.wrap_cnn_for_art(
        model, n_features=X_train.shape[1], n_classes=n_classes, device=DEVICE,
    )
    out = {}
    clean_pred = clf.predict(X_test_f).argmax(axis=1)
    out["clean"] = f1_score(y_test, clean_pred, average=None,
                            labels=range(n_classes), zero_division=0)
    for eps in config.FGSM_EPSILONS:
        X_adv = attacks.generate_pgd(clf, X_test_f, epsilon=eps)
        pred = clf.predict(X_adv).argmax(axis=1)
        out[eps] = f1_score(y_test, pred, average=None,
                            labels=range(n_classes), zero_division=0)
    return out

evals = {name: per_class_under_pgd(m) for name, m in models_to_eval.items()}

# Print per-class comparison at each epsilon
eps_keys = ["clean"] + list(config.FGSM_EPSILONS)
order = sorted(range(n_classes), key=lambda i: -diversity[class_names[i]])

for eps in eps_keys:
    print(f"\n=== PGD eps={eps} : per-class F1 ===")
    print(f"{'class':20s} {'sigs':>7} {'baseline':>10} {'def_mean':>10} {'def_full':>10}")
    for i in order:
        name = class_names[i]
        b = evals['baseline'][eps][i]
        m = evals['def_meaning'][eps][i]
        f = evals['def_full'][eps][i]
        print(f"{name:20s} {diversity[name]:>7} {b:>10.3f} {m:>10.3f} {f:>10.3f}")

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]


=== PGD eps=clean : per-class F1 ===
class                   sigs   baseline   def_mean   def_full
benign                 21188      0.998      0.995      0.990
max-speedometer        10559      1.000      1.000      0.999
reverse-light-on        5994      0.999      0.997      0.996
reverse-light-off       1525      0.977      0.944      0.896
fuzzing                  592      0.996      0.996      0.996

=== PGD eps=0.01 : per-class F1 ===
class                   sigs   baseline   def_mean   def_full
benign                 21188      0.934      0.930      0.951
max-speedometer        10559      1.000      1.000      0.971
reverse-light-on        5994      0.678      0.850      0.753
reverse-light-off       1525      0.722      0.000      0.120
fuzzing                  592      0.996      0.996      0.996

=== PGD eps=0.05 : per-class F1 ===
class                   sigs   baseline   def_mean   def_full
benign                 21188      0.788      0.703      0.665
max-speedometer     

## Cross-validated defence comparison (5-fold, PGD)

Repeats the baseline vs meaningful-defence vs full-defence comparison across 5
folds so each number carries a mean and standard deviation. Given the known
fold-to-fold instability of the small classes, this is the only trustworthy
version of the defence result. Trains 3 models per fold (the two defences train
twice internally), so this is the most compute-heavy step in the project.

In [5]:
import importlib, crossval
importlib.reload(crossval)
print("crossval_defence_comparison" in dir(crossval))

True


In [6]:
import pandas as pd

road_strict = pd.read_csv(config.PROCESSED_DIR / "road_strict.csv")

defence_cv = crossval.crossval_defence_comparison(
    road_strict,
    config.FEATURE_COLUMNS,
    class_names,
    epsilons=config.FGSM_EPSILONS,
    eps_meaningful=config.ROAD_DEFENCE_EPS_MEANINGFUL,
    eps_full=config.ROAD_DEFENCE_EPS_FULL,
    n_splits=5,
    dup_target=200,
    device=DEVICE,
    random_seed=config.RANDOM_SEED,
    cnn_epochs=50,
)
print("\nDefence CV complete.")

Label mapping:
    0 -> benign
    1 -> fuzzing
    2 -> max-speedometer
    3 -> reverse-light-off
    4 -> reverse-light-on
    epoch   1/50     loss 0.4474
    epoch   5/50     loss 0.0532
    epoch  10/50     loss 0.0177
    epoch  15/50     loss 0.0112
    epoch  20/50     loss 0.0115
    epoch  25/50     loss 0.0138
    epoch  30/50     loss 0.0079
    epoch  35/50     loss 0.0078
    epoch  40/50     loss 0.0056
    epoch  45/50     loss 0.0072
    epoch  50/50     loss 0.0096
    epoch   1/50     loss 0.3901
    epoch   5/50     loss 0.0366
    epoch  10/50     loss 0.0228
    epoch  15/50     loss 0.0135
    epoch  20/50     loss 0.0119
    epoch  25/50     loss 0.0109
    epoch  30/50     loss 0.0099
    epoch  35/50     loss 0.0092
    epoch  40/50     loss 0.0074
    epoch  45/50     loss 0.0062
    epoch  50/50     loss 0.0105


PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/997 [00:00<?, ?it/s]

    augmented trainset: 31886 clean -> 127544 total (pgd strategy)
    epoch   1/50     loss 0.1993
    epoch   5/50     loss 0.0474
    epoch  10/50     loss 0.0202
    epoch  15/50     loss 0.0152
    epoch  20/50     loss 0.0127
    epoch  25/50     loss 0.0143
    epoch  30/50     loss 0.0112
    epoch  35/50     loss 0.0117
    epoch  40/50     loss 0.0111
    epoch  45/50     loss 0.0115
    epoch  50/50     loss 0.0122
    epoch   1/50     loss 0.3900
    epoch   5/50     loss 0.0337
    epoch  10/50     loss 0.0209
    epoch  15/50     loss 0.0127
    epoch  20/50     loss 0.0115
    epoch  25/50     loss 0.0117
    epoch  30/50     loss 0.0092
    epoch  35/50     loss 0.0103
    epoch  40/50     loss 0.0062
    epoch  45/50     loss 0.0129
    epoch  50/50     loss 0.0127


...

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

  fold 5/5 done

Defence CV complete.


In [7]:
import numpy as np

diversity = {
    "benign": 21188, "max-speedometer": 10559, "reverse-light-on": 5994,
    "reverse-light-off": 1525, "fuzzing": 592,
}
n_classes = len(class_names)
eps_keys = ["clean"] + list(config.FGSM_EPSILONS)
order = sorted(range(n_classes), key=lambda i: -diversity[class_names[i]])

def cell(model, eps, i):
    s = np.array(defence_cv[model][eps][i])
    return f"{s.mean():.2f}\u00b1{s.std():.2f}"

for eps in eps_keys:
    print(f"\n=== PGD eps={eps} : per-class F1 (mean\u00b1std, 5 folds) ===")
    print(f"{'class':20s} {'sigs':>7} {'baseline':>12} {'def_mean':>12} {'def_full':>12}")
    for i in order:
        name = class_names[i]
        print(f"{name:20s} {diversity[name]:>7} "
              f"{cell('baseline', eps, i):>12} {cell('def_meaning', eps, i):>12} "
              f"{cell('def_full', eps, i):>12}")


=== PGD eps=clean : per-class F1 (mean±std, 5 folds) ===
class                   sigs     baseline     def_mean     def_full
benign                 21188    1.00±0.00    1.00±0.00    1.00±0.00
max-speedometer        10559    1.00±0.00    1.00±0.00    1.00±0.00
reverse-light-on        5994    1.00±0.00    1.00±0.01    1.00±0.00
reverse-light-off       1525    0.97±0.01    0.97±0.01    0.97±0.01
fuzzing                  592    1.00±0.00    1.00±0.00    1.00±0.00

=== PGD eps=0.01 : per-class F1 (mean±std, 5 folds) ===
class                   sigs     baseline     def_mean     def_full
benign                 21188    0.95±0.02    0.93±0.04    0.91±0.03
max-speedometer        10559    1.00±0.00    1.00±0.00    0.95±0.06
reverse-light-on        5994    0.78±0.09    0.74±0.10    0.78±0.11
reverse-light-off       1525    0.73±0.06    0.03±0.03    0.02±0.02
fuzzing                  592    1.00±0.00    1.00±0.00    1.00±0.00

=== PGD eps=0.05 : per-class F1 (mean±std, 5 folds) ===
class       

In [8]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))
import realism
print([f for f in dir(realism) if not f.startswith("_")])

['learn_observed_ranges', 'np', 'observed_range_mask', 'round_to_integer_frames']
